# Hybrid Fusion: Combining Transformer and Lexicon-Based Scores

This notebook combines the DistilBERT baseline outputs with snippet/lexicon scores
to create a hybrid sentiment representation. The goal is to balance transformer confidence
with rule-based interpretability.

In [19]:
# 1. Setup
import pandas as pd
import re
# Load transformer baseline predictions (5-class)
baseline = pd.read_csv("../data/processed/distilbert_baseline_5class.csv")

# Load snippet/lexicon scores (5-class)
snippet = pd.read_csv("../data/processed/snippet_scores_5class.csv")

print(f"✅ Loaded Baseline predictions: {baseline.shape}")
print(f"✅ Loaded Snippet scores: {snippet.shape}")

✅ Loaded Baseline predictions: (14166, 6)
✅ Loaded Snippet scores: (14166, 8)


In [20]:
# 2. Quick sanity checks
print("Baseline columns:", baseline.columns.tolist())
print("Snippet columns:", snippet.columns.tolist())

print("Companies:", snippet['company'].unique())
print("Years:", snippet['year'].unique())

Baseline columns: ['company', 'year', 'sentence', 'label', 'score', 'sentiment_5class']
Snippet columns: ['company', 'year', 'sentence', 'label', 'score', 'sentiment_5class', 'lexicon_score', 'lexicon_5class']
Companies: ['Google' 'HSBC' 'Nestle']
Years: [2022 2023 2024]


In [21]:
import re

# === 2. Custom ESG Lexicon + Scorer ===
ESG_LEXICON = {
    "positive": [
        "sustainable", "renewable", "green", "inclusive", "responsible",
        "net", "zero", "diversity", "environmental", "governance", "social",
        "ethical", "recycling", "efficiency", "compliance", "innovation",
        "equity", "fairness", "biodiversity", "community", "wellbeing"
    ],
    "negative": [
        "emission", "emissions", "pollution", "scandal", "deforestation",
        "fine", "controversy", "risk", "hazard", "lawsuit", "waste",
        "shortage", "violation", "fraud", "breach", "exploitation",
        "child", "forced", "toxic", "unethical"
    ]
}

NEGATIONS = ["no", "not", "never", "none", "without"]
INTENSIFIERS = ["very", "highly", "extremely", "significantly"]

def calculate_esg_lexicon_score(sentence: str) -> float:
    words = re.findall(r"\w+", sentence.lower())
    score = 0
    count = 0
    
    for i, word in enumerate(words):
        multiplier = 1.0
        if i > 0 and words[i-1] in INTENSIFIERS:
            multiplier = 1.5
        if word in ESG_LEXICON["positive"]:
            score += 1.0 * multiplier
            count += 1
        if word in ESG_LEXICON["negative"]:
            score -= 1.0 * multiplier
            count += 1
        if i > 0 and words[i-1] in NEGATIONS:
            score *= -1
    
    if count > 0:
        score = max(min(score / count, 1.0), -1.0)
    else:
        score = 0.0
    return score

In [22]:
# --- 3. Doc-level Lexicon aggregation (updated) ---

bert_doc = (
    baseline.groupby(["company", "year"])
    .agg({
        "score": "mean"  # BERT confidence score
    })
    .reset_index()
    .rename(columns={"score": "bert_mean_score"})
)


from collections import defaultdict


lexicon_scores = []
for _, row in baseline.iterrows(): 
    company, year, sentence = row['company'], row['year'], row['sentence']
    lex_score = calculate_esg_lexicon_score(sentence)  # <-- Güncelleme yapıldı
    lexicon_scores.append({
        'company': company,
        'year': year, 
        'lex_score': lex_score
    })


lexicon_df = pd.DataFrame(lexicon_scores)
snippet_doc = (
    lexicon_df.groupby(["company", "year"])
    .agg({
        "lex_score": "mean"
    })
    .reset_index()
    .rename(columns={"lex_score": "lexicon_score"})
)

print("✅ Doc-level lexicon aggregation completed")
print(snippet_doc.head())

✅ Doc-level lexicon aggregation completed
  company  year  lexicon_score
0  Google  2022       0.036127
1  Google  2023       0.091120
2  Google  2024       0.001980
3    HSBC  2022       0.038061
4    HSBC  2023      -0.015724


In [23]:
# 4. Merge Baseline (doc-level) + Lexicon (doc-level)
hybrid_df = pd.merge(snippet_doc, bert_doc, on=["company", "year"], how="inner")

print("✅ Hybrid doc-level DataFrame created:", hybrid_df.shape)
print(hybrid_df.head())


✅ Hybrid doc-level DataFrame created: (9, 4)
  company  year  lexicon_score  bert_mean_score
0  Google  2022       0.036127         0.898745
1  Google  2023       0.091120         0.937662
2  Google  2024       0.001980         0.927499
3    HSBC  2022       0.038061         0.942709
4    HSBC  2023      -0.015724         0.938589


In [24]:
# 4. Merge Baseline (doc-level) + Lexicon (doc-level)
hybrid_df = pd.merge(snippet_doc, bert_doc, on=["company", "year"], how="inner")

print("✅ Hybrid doc-level DataFrame created:", hybrid_df.shape)
print(hybrid_df.head())

# --- Normalize lexicon scores (min-max scaling) ---
min_val = hybrid_df["lexicon_score"].min()
max_val = hybrid_df["lexicon_score"].max()

hybrid_df["lexicon_score_normalized"] = hybrid_df["lexicon_score"].apply(
    lambda x: (x - min_val) / (max_val - min_val) if max_val != min_val else 0.5
)

# --- Fusion (0.7 BERT + 0.3 Lexicon) ---
hybrid_df["hybrid_score"] = (
    0.7 * hybrid_df["bert_mean_score"] + 0.3 * hybrid_df["lexicon_score_normalized"]
)

# --- 5-class mapping (0–1 scale) ---
def hybrid_to_5class(score, high=0.8, low=0.2):
    if score >= high:
        return "VERY POSITIVE"
    elif score >= 0.6:
        return "POSITIVE"
    elif score >= 0.4:
        return "NEUTRAL"
    elif score >= low:
        return "NEGATIVE"
    else:
        return "VERY NEGATIVE"

hybrid_df["hybrid_5class"] = hybrid_df["hybrid_score"].apply(hybrid_to_5class)

# --- Save final doc-level hybrid scores ---
out_path = "../data/processed/hybrid_scores_5class.csv"
hybrid_df.to_csv(out_path, index=False)

print(f"✅ Saved hybrid 5-class scores to {out_path}")
print(hybrid_df[["company","year","lexicon_score","lexicon_score_normalized","bert_mean_score","hybrid_score","hybrid_5class"]])

✅ Hybrid doc-level DataFrame created: (9, 4)
  company  year  lexicon_score  bert_mean_score
0  Google  2022       0.036127         0.898745
1  Google  2023       0.091120         0.937662
2  Google  2024       0.001980         0.927499
3    HSBC  2022       0.038061         0.942709
4    HSBC  2023      -0.015724         0.938589
✅ Saved hybrid 5-class scores to ../data/processed/hybrid_scores_5class.csv
  company  year  lexicon_score  lexicon_score_normalized  bert_mean_score  \
0  Google  2022       0.036127                  0.563682         0.898745   
1  Google  2023       0.091120                  0.932731         0.937662   
2  Google  2024       0.001980                  0.334528         0.927499   
3    HSBC  2022       0.038061                  0.576664         0.942709   
4    HSBC  2023      -0.015724                  0.215722         0.938589   
5    HSBC  2024      -0.047869                  0.000000         0.939063   
6  Nestle  2022       0.090833                  0.93

In [25]:
# === 6. Improved Sentence-level Lexicon with Negations and Intensifiers (Final) ===
import pandas as pd
import re

# --- 5-class mapping for –1..+1 lexicon scores ---
def lexicon_to_5class(score, high=0.5, low=0.05):
    if score >= high:
        return "VERY POSITIVE"
    elif score >= low:
        return "POSITIVE"
    elif score <= -high:
        return "VERY NEGATIVE"
    elif score <= -low:
        return "NEGATIVE"
    else:
        return "NEUTRAL"

# --- Custom ESG Lexicon Scorer ---
ESG_LEXICON = {
    "positive": [
        "sustainable", "renewable", "green", "inclusive", "responsible",
        "net", "zero", "diversity", "environmental", "governance", "social",
        "ethical", "recycling", "efficiency", "compliance", "innovation",
        "equity", "fairness", "biodiversity", "community", "wellbeing"
    ],
    "negative": [
        "emission", "emissions", "pollution", "scandal", "deforestation",
        "fine", "controversy", "risk", "hazard", "lawsuit", "waste",
        "shortage", "violation", "fraud", "breach", "exploitation",
        "child", "forced", "toxic", "unethical"
    ]
}

NEGATIONS = ["no", "not", "never", "none", "without"]
INTENSIFIERS = ["very", "highly", "extremely", "significantly"]

def calculate_esg_lexicon_score(sentence: str) -> float:
    words = re.findall(r"\w+", sentence.lower())
    score = 0
    count = 0
    
    for i, word in enumerate(words):
        multiplier = 1.0
        if i > 0 and words[i-1] in INTENSIFIERS:
            multiplier = 1.5
        if word in ESG_LEXICON["positive"]:
            score += 1.0 * multiplier
            count += 1
        if word in ESG_LEXICON["negative"]:
            score -= 1.0 * multiplier
            count += 1
        if i > 0 and words[i-1] in NEGATIONS:
            score *= -1
    
    if count > 0:
        score = max(min(score / count, 1.0), -1.0)
    else:
        score = 0.0
    return score

# --- Load baseline sentence-level predictions ---
baseline = pd.read_csv("../data/processed/distilbert_baseline_5class.csv")

# --- Lexicon scoring using new ESG scorer ---
baseline["lexicon_score"] = baseline["sentence"].apply(calculate_esg_lexicon_score)

# --- Map lexicon scores to 5-class (–1..1 scale) ---
baseline["lexicon_5class"] = baseline["lexicon_score"].apply(lambda x: lexicon_to_5class(x, high=0.5, low=0.05))

# --- Hybrid fusion (0.7 BERT + 0.3 Lexicon) ---
baseline["hybrid_score"] = 0.7 * baseline["score"] + 0.3 * baseline["lexicon_score"]
baseline["hybrid_5class"] = baseline["hybrid_score"].apply(lambda x: lexicon_to_5class(x, high=0.5, low=0.05))

# --- Save updated sentence-level CSVs ---
baseline.to_csv("../data/processed/lexicon_5class_sentencelevel.csv", index=False)
baseline.to_csv("../data/processed/hybrid_5class_sentencelevel.csv", index=False)

print("💾 Saved: lexicon_5class_sentencelevel.csv & hybrid_5class_sentencelevel.csv (updated with new ESG lexicon)")
print(baseline[["sentence","lexicon_score","lexicon_5class","hybrid_score","hybrid_5class"]].head())

💾 Saved: lexicon_5class_sentencelevel.csv & hybrid_5class_sentencelevel.csv (updated with new ESG lexicon)
                                            sentence  lexicon_score  \
0  environmental report table of contents 1 about...            1.0   
1      it also mentions notable targets set in 2022.            0.0   
2  this report outlines how we're driving positiv...            1.0   
3  for more information about our sustainability ...            0.0   
4  for more information about our overall corpora...            0.0   

  lexicon_5class  hybrid_score  hybrid_5class  
0  VERY POSITIVE      0.931467  VERY POSITIVE  
1        NEUTRAL      0.680859  VERY POSITIVE  
2  VERY POSITIVE      0.999484  VERY POSITIVE  
3        NEUTRAL      0.430939       POSITIVE  
4        NEUTRAL      0.640089  VERY POSITIVE  
